Exercise 3 — Interview Transcript to Candidate Scorecard Pipeline
Build a 3-step pipeline. Step 1 extracts the candidate's answers to each interview question from a raw transcript. Step 2 evaluates each answer against a given set of skills, flagging any skill that wasn't clearly demonstrated. Step 3 formats the evaluation into a structured hiring scorecard with an overall recommendation.

Input: transcript_text, skills_to_assess

Output: the extracted answers from Step 1, the flagged skill evaluation from Step 2, and the final scorecard from Step 3

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (field: list of question + answer pairs)

Step 2 — Template: ChatPromptTemplate (system persona: "technical interviewer evaluating skill evidence")  |  Parser: PydanticOutputParser (fields: skill, demonstrated: bool, evidence_or_gap)

Step 3 — Template: ChatPromptTemplate (system persona: "hiring panel coordinator")  |  Parser: PydanticOutputParser (fields: list of scorecard rows + overall_recommendation)

## Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (field: list of question + answer pairs)


In [1]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
class InterviewAnswer(BaseModel):
    question: str = Field(
        description="The interview question asked"
    )

    answer: str = Field(
        description="The candidate's answer to the question"
    )


class ExtractedAnswers(BaseModel):
    answers: list[InterviewAnswer] = Field(
        description="All identifiable interview question-answer pairs"
    )

In [3]:
answers_parser = PydanticOutputParser(
    pydantic_object=ExtractedAnswers
)

In [4]:
answers_prompt = PromptTemplate(
    template="""
You are extracting interview question-answer pairs.

Extract each identifiable interviewer question and the
candidate's corresponding answer from the transcript.

Rules:
- Preserve the meaning of each answer.
- Do not invent missing answers.
- Do not evaluate the candidate.
- If an answer is incomplete, preserve what is available.
- Ignore unrelated conversation that is not part of a
  question-answer pair.

Interview transcript:
{transcript_text}

{format_instructions}
""",
    input_variables=["transcript_text"],
    partial_variables={
        "format_instructions":
            answers_parser.get_format_instructions()
    }
)

In [6]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [7]:
step1_chain = (
    answers_prompt
    | llm
    | answers_parser
)

In [8]:
transcript_text = """
Interviewer: How have you used Python to solve a problem?

Candidate: I built a script to automate CSV processing.
I used pandas to clean the data and wrote tests to
check the output.

Interviewer: What is the time complexity of binary search?

Candidate: I know binary search is useful for finding
elements, but I cannot remember its time complexity.
"""

extracted_answers = step1_chain.invoke({
    "transcript_text": transcript_text
})

print(extracted_answers.model_dump_json(indent=2))

{
  "answers": [
    {
      "question": "How have you used Python to solve a problem?",
      "answer": "I built a script to automate CSV processing. I used pandas to clean the data and wrote tests to check the output."
    },
    {
      "question": "What is the time complexity of binary search?",
      "answer": "I know binary search is useful for finding elements, but I cannot remember its time complexity."
    }
  ]
}


In [9]:
for item in extracted_answers.answers:
    print("Question:", item.question)
    print("Answer:", item.answer)
    print("-" * 40)

Question: How have you used Python to solve a problem?
Answer: I built a script to automate CSV processing. I used pandas to clean the data and wrote tests to check the output.
----------------------------------------
Question: What is the time complexity of binary search?
Answer: I know binary search is useful for finding elements, but I cannot remember its time complexity.
----------------------------------------


## Step 2 — Template: ChatPromptTemplate (system persona: "technical interviewer evaluating skill evidence")  |  Parser: PydanticOutputParser (fields: skill, demonstrated: bool, evidence_or_gap)


In [10]:
skills_to_assess = [
    "Python programming",
    "Problem-solving",
    "Testing and debugging",
    "Algorithmic knowledge"
]

In [11]:
class SkillEvaluation(BaseModel):
    skill: str = Field(
        description="The skill being assessed"
    )

    demonstrated: bool = Field(
        description="True only when the transcript provides clear supporting evidence"
    )

    evidence_or_gap: str = Field(
        description="Specific supporting evidence from the answers, or what evidence is missing"
    )


class SkillEvaluationResults(BaseModel):
    evaluations: list[SkillEvaluation] = Field(
        description="Evaluation results for all requested skills"
    )

In [12]:
evaluation_parser = PydanticOutputParser(
    pydantic_object=SkillEvaluationResults
)

In [19]:

evaluation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a technical interviewer evaluating skill evidence.

Evaluate every requested skill using only the extracted
interview answers.

Rules:
- Assess every skill in the supplied list.
- Mark demonstrated=True only when clear, relevant evidence
  appears in the answers.
- Mark demonstrated=False when the evidence is absent,
  insufficient, or does not demonstrate the skill.
- Cite specific answers in evidence_or_gap.
- Do not invent evidence.
- Do not infer competence from personal characteristics.
- Lack of evidence means the skill was not demonstrated
  in this transcript; it does not prove the candidate
  lacks the skill.
- Return exactly one evaluation for each requested skill.

{format_instructions}
"""
    ),
    (
        "user",
        """
Skills to assess:
{skills_to_assess}

Extracted interview answers:
{extracted_answers}
"""
    )
]).partial(
    format_instructions=evaluation_parser.get_format_instructions()
)


In [20]:
ex3_step2_chain = (
    evaluation_prompt
    | llm
    | evaluation_parser
)

In [21]:
skills_to_assess = [
    "Python programming",
    "Problem-solving",
    "Testing and debugging",
    "Algorithmic knowledge"
]

skill_evaluations = ex3_step2_chain.invoke({
    "skills_to_assess": "\n".join(
        f"- {skill}" for skill in skills_to_assess
    ),
    "extracted_answers":
        extracted_answers.model_dump_json(indent=2)
})

print(skill_evaluations.model_dump_json(indent=2))

{
  "evaluations": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Built a script to automate CSV processing using pandas."
    },
    {
      "skill": "Problem-solving",
      "demonstrated": true,
      "evidence_or_gap": "Created a script to automate CSV processing, solving a data handling problem."
    },
    {
      "skill": "Testing and debugging",
      "demonstrated": true,
      "evidence_or_gap": "Wrote tests to check the output of the CSV processing script; no explicit debugging evidence."
    },
    {
      "skill": "Algorithmic knowledge",
      "demonstrated": false,
      "evidence_or_gap": "Unable to recall the time complexity of binary search."
    }
  ]
}


In [22]:
for evaluation in skill_evaluations.evaluations:
    print("Skill:", evaluation.skill)
    print("Demonstrated:", evaluation.demonstrated)
    print("Evidence or gap:", evaluation.evidence_or_gap)
    print("-" * 40)

Skill: Python programming
Demonstrated: True
Evidence or gap: Built a script to automate CSV processing using pandas.
----------------------------------------
Skill: Problem-solving
Demonstrated: True
Evidence or gap: Created a script to automate CSV processing, solving a data handling problem.
----------------------------------------
Skill: Testing and debugging
Demonstrated: True
Evidence or gap: Wrote tests to check the output of the CSV processing script; no explicit debugging evidence.
----------------------------------------
Skill: Algorithmic knowledge
Demonstrated: False
Evidence or gap: Unable to recall the time complexity of binary search.
----------------------------------------


## Step 3 — Template: ChatPromptTemplate (system persona: "hiring panel coordinator")  |  Parser: PydanticOutputParser (fields: list of scorecard rows + overall_recommendation)



In [23]:
class ScorecardRow(BaseModel):
    skill: str = Field(
        description="The skill being assessed"
    )

    demonstrated: bool = Field(
        description="Whether the supplied evaluation found clear evidence"
    )

    evidence_or_gap: str = Field(
        description="The evidence or missing evidence for this skill"
    )


class CandidateScorecard(BaseModel):
    scorecard_rows: list[ScorecardRow] = Field(
        description="Structured rows covering the skills assessed"
    )

    overall_recommendation: str = Field(
        description="A cautious recommendation for human review, with supporting rationale and any limitations"
    )

In [24]:
scorecard_parser = PydanticOutputParser(
    pydantic_object=CandidateScorecard
)

In [29]:

scorecard_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a hiring panel coordinator.

Prepare a structured interview scorecard from the supplied
skill evaluations.

Rules:
- Include every evaluated skill in the scorecard.
- Preserve the demonstrated status and supporting evidence.
- Base the overall recommendation on the supplied evidence.
- Identify important evidence gaps and limitations.
- Do not invent candidate qualifications or interview results.
- Do not treat missing evidence as proof of inability.
- The recommendation is advisory and must be reviewed by
  the human hiring panel.
- Do not make assumptions about protected or personal
  characteristics.
- Follow the required output schema.

{format_instructions}
"""
    ),
    (
        "user",
        """
Skill evaluations:
{skill_evaluations}

Prepare the final structured candidate scorecard.
"""
    )
]).partial(
    format_instructions=scorecard_parser.get_format_instructions()
)


In [30]:
ex3_step3_chain = (
    scorecard_prompt
    | llm
    | scorecard_parser
)

In [31]:
candidate_scorecard = ex3_step3_chain.invoke({
    "skill_evaluations":
        skill_evaluations.model_dump_json(indent=2)
})

print(candidate_scorecard.model_dump_json(indent=2))

{
  "scorecard_rows": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Built a script to automate CSV processing using pandas."
    },
    {
      "skill": "Problem-solving",
      "demonstrated": true,
      "evidence_or_gap": "Created a script to automate CSV processing, solving a data handling problem."
    },
    {
      "skill": "Testing and debugging",
      "demonstrated": true,
      "evidence_or_gap": "Wrote tests to check the output of the CSV processing script; no explicit debugging evidence."
    },
    {
      "skill": "Algorithmic knowledge",
      "demonstrated": false,
      "evidence_or_gap": "Unable to recall the time complexity of binary search."
    }
  ],
  "overall_recommendation": "The candidate shows clear evidence of proficiency in Python programming, problem‑solving, and testing. However, there is a notable gap in algorithmic knowledge, and the debugging evidence is limited to a single instance. Recommend a foc

In [32]:
for row in candidate_scorecard.scorecard_rows:
    print("Skill:", row.skill)
    print("Demonstrated:", row.demonstrated)
    print("Evidence or gap:", row.evidence_or_gap)
    print("-" * 40)

print("OVERALL RECOMMENDATION:")
print(candidate_scorecard.overall_recommendation)

Skill: Python programming
Demonstrated: True
Evidence or gap: Built a script to automate CSV processing using pandas.
----------------------------------------
Skill: Problem-solving
Demonstrated: True
Evidence or gap: Created a script to automate CSV processing, solving a data handling problem.
----------------------------------------
Skill: Testing and debugging
Demonstrated: True
Evidence or gap: Wrote tests to check the output of the CSV processing script; no explicit debugging evidence.
----------------------------------------
Skill: Algorithmic knowledge
Demonstrated: False
Evidence or gap: Unable to recall the time complexity of binary search.
----------------------------------------
OVERALL RECOMMENDATION:
The candidate shows clear evidence of proficiency in Python programming, problem‑solving, and testing. However, there is a notable gap in algorithmic knowledge, and the debugging evidence is limited to a single instance. Recommend a focused follow‑up interview or assessment to